# Image Transformations with NumPy
## Complete Solution Notebook (Expanded Edition)

This notebook contains **full solutions**, **alternate implementations**, advanced techniques, a rich simulation section, and quantitative metrics. Use it to check your work or for reference.

All outputs are printed/shown so you can verify results immediately.

## Learning Objectives & Project Flowchart
Same objectives as the skeleton. The flowchart below shows the complete journey we will take.

In [ ]:
# Full workflow flowchart (identical in both notebooks)

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import numpy as np

fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(0, 14)
ax.set_ylim(0, 9)
ax.axis('off')
ax.set_title("Image Transformations with NumPy — Project Workflow Flowchart", fontsize=16, fontweight='bold', pad=20)

# Colors
box_color = "#E3F2FD"
alt_color = "#FFF3E0"
advanced_color = "#E8F5E9"
la_color = "#F3E5F5"
sim_color = "#FFEBEE"
end_color = "#E0F7FA"

def add_box(ax, x, y, w, h, text, color=box_color, fontsize=9):
    box = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.03,rounding_size=0.2",
                         facecolor=color, edgecolor='#37474F', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=fontsize,
            wrap=True, fontweight='medium')

def add_arrow(ax, start, end, color='#455A64'):
    ax.annotate('', xy=end, xytext=start,
                arrowprops=dict(arrowstyle='->', color=color, lw=1.8,
                               connectionstyle='arc3,rad=0.05'))

# Row 1 - Setup
add_box(ax, 0.5, 7.5, 2.8, 1.0, "1. Setup & Imports\nnumpy + matplotlib\nDefine base image(s)\nshow_image() helper", box_color)
add_box(ax, 3.8, 7.5, 2.8, 1.0, "2. Understand Image\nas NumPy Array\nInspect shape, dtype,\npixel values & visualize", box_color)
add_box(ax, 7.1, 7.5, 2.8, 1.0, "3. Basic Transforms\n• Invert colors (3 ways)\n• Flip (h/v) & Transpose\n• Rotate 90/180/270", alt_color)
add_box(ax, 10.4, 7.5, 3.1, 1.0, "4. Intensity & Noise\n• Brightness / Contrast\n• Add Gaussian noise\n• Salt & pepper", advanced_color)

# Arrows row 1
add_arrow(ax, (3.3, 8.0), (3.8, 8.0))
add_arrow(ax, (6.6, 8.0), (7.1, 8.0))
add_arrow(ax, (9.9, 8.0), (10.4, 8.0))

# Row 2 - Advanced
add_box(ax, 0.5, 5.3, 3.5, 1.2, "5. Filtering & Edges (Advanced)\n• Gaussian blur (scipy.ndimage)\n• Simple convolution kernels\n• Edge detection hint (Sobel-like)", advanced_color)
add_box(ax, 4.5, 5.3, 4.5, 1.2, "6. Linear Algebra Power\nSolve: random @ X = heart_img\nUse np.linalg.solve\nReconstruct & check error\n(Concept: system of equations)", la_color)
add_box(ax, 9.5, 5.3, 4.0, 1.2, "7. Simulation Section\nChange params:\nbrightness, noise_level, flip\nSee metrics: MSE, mean, std\nVisual comparison grid", sim_color)

add_arrow(ax, (2.25, 7.5), (2.25, 6.5))
add_arrow(ax, (6.85, 7.5), (6.85, 6.5))
add_arrow(ax, (11.95, 7.5), (11.5, 6.5))

# Row 3 - Practice & Insights
add_box(ax, 0.5, 3.0, 4.0, 1.3, "8. More Practice Exercises\n• Create custom shape (star/letter)\n• Chain transforms creatively\n• Recover from noisy observation\n• Compare vectorized vs loops", box_color)
add_box(ax, 5.0, 3.0, 4.5, 1.3, "9. Key Insights & Audience\n• Images = matrices = data\n• Linear algebra in real world\n• Consider audience data literacy\n  when presenting visuals", end_color)
add_box(ax, 10.0, 3.0, 3.5, 1.3, "10. Next Steps\nscikit-image, OpenCV\nPyTorch transforms\nReal image pipelines\nData augmentation for ML", end_color)

add_arrow(ax, (2.25, 5.3), (2.5, 4.3))
add_arrow(ax, (6.75, 5.3), (7.25, 4.3))
add_arrow(ax, (11.5, 5.3), (11.75, 4.3))

# Bottom summary
add_box(ax, 3.5, 0.8, 7.0, 1.2, "🎯 Desired Outcome: Build intuition for matrix operations on visual data,\nmaster multiple ways to achieve same transform, explore parameter impact via simulation,\nand understand practical uses in data pipelines & computer vision preprocessing.",
        "#FFFDE7", fontsize=10)

# Connecting arrows to bottom
add_arrow(ax, (2.5, 3.0), (5.0, 2.0))
add_arrow(ax, (7.25, 3.0), (7.0, 2.0))
add_arrow(ax, (11.75, 3.0), (9.0, 2.0))

plt.tight_layout()
plt.show()
print("Flowchart generated successfully! This visual summarizes the learning journey.")


## 1. Setup, Imports & Helper

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
import warnings
warnings.filterwarnings('ignore')

def show_image(image, title="Image", cmap="gray", figsize=(5,5)):
    """Improved helper: shows image + prints key statistics"""
    plt.figure(figsize=figsize)
    plt.imshow(np.clip(image, 0, 255), cmap=cmap)  # safety clip
    plt.title(title, fontsize=12)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    print(f"[{title}] shape={image.shape}, dtype={image.dtype}, "
          f"min={float(image.min()):.2f}, max={float(image.max()):.2f}, "
          f"mean={float(image.mean()):.2f}, std={float(image.std()):.2f}")
    return image   # allow chaining if desired

print("✅ All libraries and helper ready.")

In [ ]:
# Base heart image (float for easy math)
heart_img = np.array([[255,   0,   0, 255,   0,   0, 255],
                      [  0, 127.5, 127.5,   0, 127.5, 127.5,   0],
                      [  0, 127.5, 127.5, 127.5, 127.5, 127.5,   0],
                      [  0, 127.5, 127.5, 127.5, 127.5, 127.5,   0],
                      [255,   0, 127.5, 127.5, 127.5,   0, 255],
                      [255, 255,   0, 127.5,   0, 255, 255],
                      [255, 255, 255,   0, 255, 255, 255]], dtype=float)

show_image(heart_img, "Original Heart Image")

## 2. Understanding the Array (with printed inspection)

In [ ]:
print("=== ARRAY INSPECTION ===")
print("Shape:", heart_img.shape)
print("dtype:", heart_img.dtype)
print("Center (3,3):", heart_img[3,3])
print("Top row   :", heart_img[0])
print("Left col  :", heart_img[:,0])
print("\nNotice how the visual heart emerges from the numeric pattern!")

## 3. Basic Transformations — Multiple Alternate Implementations

### 3.1 Color Inversion — Three Different Ways

In [ ]:
print("=== INVERSION ALTERNATIVES ===")
# Method 1: Simple arithmetic (recommended for float images)
inv1 = 255 - heart_img
show_image(inv1, "Invert Method 1: 255 - img")

# Method 2: Using numpy invert (best on uint8)
inv2 = np.invert(heart_img.astype(np.uint8)).astype(float)
show_image(inv2, "Invert Method 2: np.invert (uint8)")

# Method 3: Max - img (robust to any range)
inv3 = heart_img.max() - heart_img
show_image(inv3, "Invert Method 3: max() - img")

print("All three produce visually identical results (minor floating point diffs possible).")

### 3.2 Geometric Transforms — Flip, Transpose, Rotate

In [ ]:
print("=== GEOMETRIC TRANSFORMS ===")
show_image(np.fliplr(heart_img), "Horizontal Flip (fliplr)")
show_image(np.flipud(heart_img), "Vertical Flip (flipud)")
show_image(heart_img.T, "Transpose (.T) — Original 'Rotate'")
show_image(np.rot90(heart_img, k=3), "90° Counter-Clockwise (rot90 k=3)")

print("Note: For square matrices, transpose + flips can achieve any 90° rotation.")

## 4. Intensity, Noise & Practical Augmentation

In [ ]:
print("=== BRIGHTNESS & NOISE ===")
# Brightness with clipping (important to avoid overflow)
for factor in [0.6, 1.0, 1.8]:
    b = np.clip(heart_img * factor, 0, 255)
    show_image(b, f"Brightness ×{factor}")

# Noise at different levels
np.random.seed(42)
for std in [10, 40, 90]:
    noisy = np.clip(heart_img + np.random.normal(0, std, heart_img.shape), 0, 255)
    show_image(noisy, f"Gaussian Noise σ={std}")

## 5. Advanced Filtering — Blur & Feature Enhancement

In [ ]:
print("=== FILTERING ===")
for sigma in [0.5, 1.2, 2.5]:
    blurred = ndimage.gaussian_filter(heart_img, sigma=sigma)
    show_image(blurred, f"Gaussian Blur σ={sigma}")
    diff = blurred.mean() - heart_img.mean()
    print(f"   Mean shift after blur: {diff:+.2f}")

# Simple edge emphasis (high-pass like)
edge_kernel = np.array([[-1,-1,-1],[-1,8,-1],[-1,-1,-1]])
edges = ndimage.convolve(heart_img, edge_kernel)
show_image(np.clip(edges, 0, 255), "Simple Edge Emphasis (manual conv)")

## 6. Linear Algebra: Image Recovery via Matrix Solve

In [ ]:
print("=== LINEAR ALGEBRA RECOVERY ===")
np.random.seed(123)
A = np.random.randint(10, 240, size=(7,7)).astype(float)   # random 'mixing' matrix
show_image(A, "Random Matrix A")

X = np.linalg.solve(A, heart_img)          # solve A @ X = heart_img
show_image(X, "Solved coefficient matrix X")

reconstructed = A @ X
show_image(reconstructed, "Reconstructed = A @ X (should ≈ original)")

error = np.linalg.norm(reconstructed - heart_img)
print(f"Frobenius reconstruction error: {error:.6f}")
print("✅ Perfect recovery (within machine precision) because A is square & invertible.")

# Bonus: what if A is noisy? Use least squares
A_noisy = A + np.random.normal(0, 5, A.shape)
X_ls, residuals, rank, s = np.linalg.lstsq(A_noisy, heart_img, rcond=None)
rec_ls = A_noisy @ X_ls
print(f"Least-squares error on noisy A: {np.linalg.norm(rec_ls - heart_img):.2f}")

## 7. Simulation & Parameter Exploration (Fully Interactive by Re-running)

In [ ]:
# SIMULATION DASHBOARD — Change values and re-run the whole cell
print("\n" + "="*60)
print("SIMULATION DASHBOARD — Modify parameters above the line")
print("="*60)

# === USER CONTROLS ===
brightness = 1.25
noise_level = 22
blur_sigma = 0.9
invert = False
flip = None          # None, 'h', or 'v'
add_edge = False

# === PIPELINE ===
img = heart_img.copy()
img = np.clip(img * brightness, 0, 255)
if noise_level > 0:
    img = np.clip(img + np.random.normal(0, noise_level, img.shape), 0, 255)
if blur_sigma > 0:
    img = ndimage.gaussian_filter(img, sigma=blur_sigma)
if invert:
    img = 255 - img
if flip == 'h':
    img = np.fliplr(img)
elif flip == 'v':
    img = np.flipud(img)
if add_edge:
    edge_k = np.array([[-1,-1,-1],[-1,8,-1],[-1,-1,-1]])
    img = np.clip(ndimage.convolve(img, edge_k), 0, 255)

# === METRICS ===
orig = heart_img
mae = np.mean(np.abs(img - orig))
mse = np.mean((img - orig)**2)
rmse = np.sqrt(mse)
psnr = 20 * np.log10(255.0 / rmse) if rmse > 0 else np.inf
mean_diff = img.mean() - orig.mean()
std_diff = img.std() - orig.std()

print(f"brightness={brightness} | noise_σ={noise_level} | blur_σ={blur_sigma}")
print(f"MAE  = {mae:7.2f}   (lower is better)")
print(f"RMSE = {rmse:7.2f}")
print(f"PSNR = {psnr:7.2f} dB   (higher is better, >30 excellent)")
print(f"Δmean= {mean_diff:+6.2f}   Δstd={std_diff:+6.2f}")

# Visual 2x2 grid
fig, axes = plt.subplots(2, 2, figsize=(9, 8))
axes[0,0].imshow(orig, cmap='gray'); axes[0,0].set_title('Original'); axes[0,0].axis('off')
axes[0,1].imshow(img, cmap='gray'); axes[0,1].set_title('Transformed'); axes[0,1].axis('off')
axes[1,0].imshow(np.abs(img - orig), cmap='hot'); axes[1,0].set_title('Absolute Difference'); axes[1,0].axis('off')
diff_hist = np.abs(img - orig).ravel()
axes[1,1].hist(diff_hist, bins=30, color='#E91E63', alpha=0.7)
axes[1,1].set_title('Distribution of Pixel Differences')
axes[1,1].set_xlabel('Absolute difference')
plt.tight_layout()
plt.show()
print("\nTip: Try extreme values (noise_level=120, brightness=0.3) to see when PSNR collapses.")

## 8. Practice Challenges — Sample Solutions

In [ ]:
# Challenge 1: Custom shape (plus sign + circle-ish)
custom = np.zeros((9,9))
custom[4, 2:7] = 220        # horizontal
custom[2:7, 4] = 220        # vertical
custom[1,3] = custom[1,5] = custom[7,3] = custom[7,5] = 180  # decoration
show_image(custom, "Custom Plus-like Shape")

# Apply chain: brightness + noise + blur
chained = np.clip(custom * 0.85 + np.random.normal(0, 18, custom.shape), 0, 255)
chained = ndimage.gaussian_filter(chained, sigma=0.7)
show_image(chained, "Chained transform on custom shape")

In [ ]:
# Challenge 3 example: Noisy recovery
noisy_heart = np.clip(heart_img + np.random.normal(0, 45, heart_img.shape), 0, 255)
recovered = ndimage.gaussian_filter(noisy_heart, sigma=1.1)
recovered = ndimage.median_filter(recovered, size=3)  # extra robust
show_image(noisy_heart, "Heavily Noisy Heart")
show_image(recovered, "Recovered via Gaussian + Median")
print(f"MAE before recovery: {np.mean(np.abs(noisy_heart - heart_img)):.2f}")
print(f"MAE after recovery : {np.mean(np.abs(recovered - heart_img)):.2f}")

## Key Insights & Audience Considerations

1. **Multiple paths, same destination**: Always know at least two ways to invert/flip/rotate — helps when working with different dtypes or performance constraints.
2. **Quantify everything**: 'It looks blurrier' → 'MAE increased by 12.4, PSNR dropped to 22 dB'.
3. **Audience awareness** (from the attached PDFs):
   - **Experts/Technicians**: Show the linear algebra, code, error metrics.
   - **Executives/Nonspecialists**: Show before/after visuals + one-sentence business impact ("We can programmatically clean and augment visual data for better ML models").
4. **Practical applicability**: These exact operations appear in every computer vision pipeline (OpenCV, Pillow, torchvision, albumentations).

---
## Appendix: Quick Reference Cheat Sheet

| Operation          | Code                              | Notes                          |
|--------------------|-----------------------------------|--------------------------------|
| Invert             | `255 - img` or `img.max()-img`   | Float friendly                 |
| Horizontal flip    | `np.fliplr(img)`                  | -                              |
| Transpose          | `img.T`                           | 90° rotate for square          |
| Brightness         | `np.clip(img * factor, 0, 255)`   | Always clip!                   |
| Gaussian noise     | `img + np.random.normal(0,s,shape)` | Clip after                     |
| Blur               | `ndimage.gaussian_filter(img, sigma)` | Great for denoising       |
| Solve system       | `np.linalg.solve(A, B)`           | A must be square & invertible  |
| Metrics            | `np.mean(np.abs(diff))`, `PSNR`   | Use for objective comparison   |

**You have completed the full expanded solution!**

Recommended next steps:
- Try the simulation with 5 different parameter combinations and record the PSNR table.
- Load a real small grayscale photo using PIL and apply the same pipeline.
- Extend the flowchart with your own custom block (e.g. 'FFT filtering').

Happy transforming! 🖼️➡️🔢➡️🖼️